# Data Preparation — AmazonHelp conversations

`Data/twcs.csv` → group into conversations → filter → `Data/amazonhelp_threads.csv`

Every discarded conversation is written out, never silently dropped. Four filters, four files: `length_discard.csv`, `gap_discard.csv`, `third_party_discard.csv`, `lang_discard.csv`. Between them they account for every conversation that entered the pipeline and did not reach the output.

Requires: Python 3.12 with `numpy`, `pandas`, `langdetect` installed (`pip install numpy pandas langdetect`).

## Assumptions

1. A conversation is a connected component of the reply graph built from `in_response_to_tweet_id`. There is no conversation id in the raw data, so it has to be reconstructed. Grouping by author instead would be wrong in both directions: it splits a thread whenever someone else replies into it, and it never reveals that the extra party was there. The graph is what makes assumption 6 possible at all — you must reconstruct the whole thread before you can tell.

2. Kept if the component contains at least one `AmazonHelp` tweet. All AmazonHelp rows are outbound, so the customer side of a ticket is kept only via the pipeline rather than assumed away.

3. Branching (non-linear) conversations are kept, and `turn_index` is assigned by `created_at`. Two caveats, both reported by the pipeline rather than assumed away:
   - For a branching thread, chronological order interleaves sibling branches, so consecutive turns are not necessarily replies to one another.
   - A small number of replies carry a timestamp earlier than the tweet they reply to, so for those conversations chronological order is not the reply chain at all. The count is printed.

   In both cases `in_response_to_tweet_id` is authoritative and is retained in the output; `is_linear` flags which conversations are plain chains.

4. DISCARD: conversations with more than 5 tweets → `length_discard.csv`.

5. DISCARD: conversations where any gap between consecutive turns reaches 24 hours → `gap_discard.csv`. Every kept conversation therefore runs end to end in under a day.

   Reply-edge adjacency does not imply conversational continuity. A stranger can reply to an old tweet and union-find will weld them into one component. A real example found in the output before this filter existed: a 2014 affiliate-spam tweet for a GE water filter → a 2016 comment about a Samsung fridge → a 2017 warranty complaint → AmazonHelp's reply. Four authors, 3.4 years, one shared word ("refrigerator"), and AmazonHelp was only ever answering the last turn.

   24 hours matches how this channel behaves — measured on an earlier run, 97.1% of conversations already completed within a day and the median largest gap was about 17 minutes. **Cost: this is the strictest threshold considered**, dropping roughly 3% of conversations rather than the ~0.6% for a 7-day cap. The extra ~2.4% are threads where a customer genuinely replied a few days later. Inspect `max_gap_hours` in `gap_discard.csv` to judge that trade-off.

6. DISCARD: conversations involving more than one customer → `third_party_discard.csv`. A kept conversation is well defined — there are two problems and one of them is being answered. The canonical example is the conversation this notebook was originally built against: `771286` reports the Kindle app losing their page, AmazonHelp replies, and then a different user `771287` chimes in with "I'm having the same problem". Structurally one thread, but two complainants.

   **Cost: about 2% of conversations** (roughly 940), and they are not random — they are the threads that attracted enough attention for strangers to join, so they skew toward widely-shared problems. Conversations with no customer tweet at all are also removed here; the count is printed separately.

7. DISCARD: non-English conversations → `lang_discard.csv`. `langdetect` runs on the concatenated customer text of each conversation, after stripping `@mentions`, URLs, `#hashtags` and agent initials. One verdict per conversation, so a conversation is never mixed — AmazonHelp replies are short and templated, so they detect unreliably.

8. Filters run cheapest-first — length, gap, third-party, then language — so the four discard files are disjoint. A conversation lands in exactly one, by whichever filter rejects it first. The slow language pass therefore only runs on conversations that already survived the other three.

9. Text is not normalized. Agent initials (`^JS`), links, mentions and hashtags stay in `text`; stripping happens only on a throwaway copy used for detection.

10. A tweet whose parent is absent from the file becomes its own conversation root, so no text is lost. Note such a parent has no timestamp to compare against, so it is excluded from the ordering check in the next cell — comparing against NaT would silently read as a violation.

**On reading the output:** `conv_id` is numbered by each conversation's earliest tweet, so the oldest — and therefore oddest — conversations sort to the top of the CSV. The first few rows are not representative of the corpus. Check the distributions, not row 1.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from langdetect import DetectorFactory, LangDetectException, detect_langs

DetectorFactory.seed = 0  # langdetect is non-deterministic without a fixed seed

DATA = (Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent) / "Data"
BRAND = "AmazonHelp"
MAX_TWEETS = 5        # assumption 4
MAX_GAP_HOURS = 24     # assumption 5 - keep only conversations whose every gap is under this
N_CUSTOMERS = 1        # assumption 6 - exactly one customer, no third-party joiners
LANG = "en"            # assumption 7
MIN_CONF = 0.0         # raise to also require a minimum langdetect probability

steps = []  # drop accounting

In [2]:
raw = pd.read_csv(
    DATA / "twcs.csv",
    dtype={"tweet_id": "int64", "author_id": "str", "text": "str",
           "response_tweet_id": "str", "in_response_to_tweet_id": "float64"},
)
raw["ts"] = pd.to_datetime(raw["created_at"], format="%a %b %d %H:%M:%S %z %Y")
assert raw["tweet_id"].is_unique

# Assumptions 1-2: union-find over reply edges. Union-find rather than a recursive walk because it is
# one linear pass and cannot overflow the stack - real chains here run hundreds of tweets deep.
index_of = {t: i for i, t in enumerate(raw["tweet_id"].to_numpy())}
parents = raw["in_response_to_tweet_id"].to_numpy()
uf = np.arange(len(raw))

def find(x):
    root = x
    while uf[root] != root:
        root = uf[root]
    while uf[x] != root:  # path compression
        uf[x], x = root, uf[x]
    return root

n_dangling = 0
for child in np.flatnonzero(~np.isnan(parents)):
    p = index_of.get(int(parents[child]))
    if p is None:
        n_dangling += 1  # assumption 10
        continue
    a, b = find(child), find(p)
    if a != b:
        uf[a] = b

raw["conv_id"] = [find(i) for i in range(len(raw))]
threads = raw[raw["conv_id"].isin(set(raw.loc[raw["author_id"] == BRAND, "conv_id"]))].copy()

steps.append(("AmazonHelp conversations", threads["conv_id"].nunique(), len(threads)))
print(f"rows {len(raw):,} | AmazonHelp tweets {(raw['author_id'] == BRAND).sum():,} | "
      f"dangling parents {n_dangling:,}")
print(f"AmazonHelp conversations {threads['conv_id'].nunique():,} | tweets {len(threads):,}")

rows 2,811,774 | AmazonHelp tweets 169,840 | dangling parents 3,862
AmazonHelp conversations 82,556 | tweets 374,042


In [3]:
# Assumption 3: turn order is chronological, so it is worth knowing how often that disagrees with the
# reply chain. Compared only where the parent is present in this set - a dangling parent (assumption 10)
# maps to NaT, and every comparison with NaT is False, which would masquerade as a violation.
kids = threads.dropna(subset=["in_response_to_tweet_id"]).copy()
kids["parent_ts"] = kids["in_response_to_tweet_id"].astype("int64").map(
    dict(zip(threads["tweet_id"], threads["ts"]))
)
known = kids.dropna(subset=["parent_ts"])
out_of_order = known[known["ts"] < known["parent_ts"]]

print(f"\nreply-vs-parent timestamps checked : {len(known):,}"
      f"  (skipped {len(kids) - len(known):,} with a parent outside this set)")
print(f"replies timestamped before parent  : {len(out_of_order):,}"
      f"  in {out_of_order['conv_id'].nunique():,} conversations")
if len(out_of_order):
    print("  -> for those conversations turn_index is chronological but NOT the reply chain;")
    print("     in_response_to_tweet_id remains authoritative.")


reply-vs-parent timestamps checked : 291,486  (skipped 654 with a parent outside this set)
replies timestamped before parent  : 0  in 0 conversations


In [4]:
# Assumption 4: discard conversations longer than MAX_TWEETS.
threads["conv_len"] = threads.groupby("conv_id")["tweet_id"].transform("size")
too_long = threads[threads["conv_len"] > MAX_TWEETS]
kept = threads[threads["conv_len"] <= MAX_TWEETS].copy()

too_long.drop(columns=["ts"]).to_csv(DATA / "length_discard.csv", index=False)
steps.append((f"length <= {MAX_TWEETS}", kept["conv_id"].nunique(), len(kept)))

print(f"discarded {too_long['conv_id'].nunique():,} conversations / {len(too_long):,} tweets "
      f"-> length_discard.csv  (longest was {threads['conv_len'].max():,} tweets)")
print(f"kept      {kept['conv_id'].nunique():,} conversations / {len(kept):,} tweets")

discarded 19,594 conversations / 190,298 tweets -> length_discard.csv  (longest was 448 tweets)
kept      62,962 conversations / 183,744 tweets


In [5]:
# Assumption 5: discard conversations welded together across long time gaps.
# A stranger replying to an old tweet forms a valid reply edge but not a real conversation.
kept = kept.sort_values(["conv_id", "ts"])
max_gap_h = (kept.groupby("conv_id")["ts"].diff().dt.total_seconds() / 3600
             ).groupby(kept["conv_id"]).max()
gap_convs = set(max_gap_h[max_gap_h >= MAX_GAP_HOURS].index)  # keep only gaps strictly under the cap

too_sparse = kept[kept["conv_id"].isin(gap_convs)].copy()
too_sparse["max_gap_hours"] = too_sparse["conv_id"].map(max_gap_h).round(1)
too_sparse.drop(columns=["ts"]).to_csv(DATA / "gap_discard.csv", index=False)

kept = kept[~kept["conv_id"].isin(gap_convs)].copy()
steps.append((f"max gap < {MAX_GAP_HOURS}h", kept["conv_id"].nunique(), len(kept)))

print(f"discarded {too_sparse['conv_id'].nunique():,} conversations / {len(too_sparse):,} tweets "
      f"-> gap_discard.csv")
print(f"kept      {kept['conv_id'].nunique():,} conversations / {len(kept):,} tweets")
print(f"\nlargest gap (hours) - median {max_gap_h.median():,.2f} | 95th pct "
      f"{max_gap_h.quantile(.95):,.1f} | max {max_gap_h.max():,.0f} ({max_gap_h.max()/24:,.0f} days)")

discarded 1,691 conversations / 6,755 tweets -> gap_discard.csv
kept      61,271 conversations / 176,989 tweets

largest gap (hours) - median 0.26 | 95th pct 10.3 | max 53,791 (2,241 days)


In [6]:
# Assumption 6: keep only conversations between ONE customer and AmazonHelp.
# A third party replying into someone else's thread makes "the customer's problem" ambiguous.
n_customers = (kept[kept["author_id"] != BRAND]
               .groupby("conv_id")["author_id"].nunique()
               .reindex(kept["conv_id"].unique(), fill_value=0))
multi_convs = set(n_customers[n_customers != N_CUSTOMERS].index)

third_party = kept[kept["conv_id"].isin(multi_convs)].copy()
third_party["n_customers"] = third_party["conv_id"].map(n_customers)
third_party.drop(columns=["ts"]).to_csv(DATA / "third_party_discard.csv", index=False)

kept = kept[~kept["conv_id"].isin(multi_convs)].copy()
steps.append((f"{N_CUSTOMERS} customer only", kept["conv_id"].nunique(), len(kept)))

print(f"discarded {third_party['conv_id'].nunique():,} conversations / {len(third_party):,} tweets "
      f"-> third_party_discard.csv")
print(f"kept      {kept['conv_id'].nunique():,} conversations / {len(kept):,} tweets")
print(f"\ndistinct customers per conversation, before this filter:")
print(n_customers.value_counts().sort_index().to_string())
print("(0 means a brand-only thread with no customer tweet - also removed here)")

discarded 1,276 conversations / 5,066 tweets -> third_party_discard.csv
kept      59,995 conversations / 171,923 tweets

distinct customers per conversation, before this filter:
author_id
1    59995
2     1200
3       75
4        1
(0 means a brand-only thread with no customer tweet - also removed here)


In [7]:
# Assumption 7: one langdetect verdict per conversation, on the customer side only.
NOISE = re.compile(r"https?://\S+|@\w+|#\w+|\^\w{1,4}\b")

def detect(text):
    """Return (lang, confidence). 'unknown' when langdetect finds no usable features."""
    try:
        top = detect_langs(NOISE.sub(" ", text or "").strip())[0]
        return top.lang, round(top.prob, 3)
    except (LangDetectException, IndexError):
        return "unknown", 0.0

customer_text = (kept[kept["inbound"] == True]  # noqa: E712
                 .groupby("conv_id")["text"].apply(lambda s: " ".join(s.dropna())))
lang = pd.DataFrame([detect(t) for t in customer_text],
                     index=customer_text.index, columns=["lang", "lang_conf"])

is_english = (lang["lang"] == LANG) & (lang["lang_conf"] >= MIN_CONF)
english_convs = set(lang.index[is_english])

discarded = kept[~kept["conv_id"].isin(english_convs)].join(lang, on="conv_id")
discarded.drop(columns=["ts"]).to_csv(DATA / "lang_discard.csv", index=False)

corpus = kept[kept["conv_id"].isin(english_convs)].copy()
steps.append((f"lang == {LANG}", corpus["conv_id"].nunique(), len(corpus)))

print(f"discarded {discarded['conv_id'].nunique():,} conversations / {len(discarded):,} tweets "
      f"-> lang_discard.csv")
print(f"kept      {corpus['conv_id'].nunique():,} conversations / {len(corpus):,} tweets\n")
print("languages detected among discards:")
print(lang.loc[~is_english, "lang"].value_counts().head(12).to_string())

discarded 15,341 conversations / 43,026 tweets -> lang_discard.csv
kept      44,654 conversations / 128,897 tweets

languages detected among discards:
lang
ja         5939
es         2916
fr         1905
de         1831
pt         1176
it          656
unknown     163
nl           77
da           71
af           57
hu           53
so           48


In [8]:
# Assumption 3: renumber conversations, then number turns chronologically.
# tweet_id breaks exact-timestamp ties so the order is deterministic across runs.
conv_map = {old: new for new, old in
            enumerate(corpus.groupby("conv_id")["ts"].min().sort_values().index, start=1)}
corpus["conv_id"] = corpus["conv_id"].map(conv_map)
corpus = corpus.sort_values(["conv_id", "ts", "tweet_id"]).reset_index(drop=True)
corpus["turn_index"] = corpus.groupby("conv_id").cumcount() + 1

# A conversation is linear when no tweet in it got more than one reply. Counted per
# (conv_id, parent), NOT globally: a parent missing from the file can be referenced from several
# different conversations, which is not branching.
branching = (corpus.dropna(subset=["in_response_to_tweet_id"])
             .groupby(["conv_id", "in_response_to_tweet_id"]).size())
branch_convs = set(branching[branching > 1].index.get_level_values("conv_id"))
corpus["is_linear"] = ~corpus["conv_id"].isin(branch_convs)

output = corpus[["conv_id", "turn_index", "tweet_id", "author_id", "inbound", "created_at", "text",
                  "in_response_to_tweet_id", "response_tweet_id", "conv_len", "is_linear"]].copy()
output["in_response_to_tweet_id"] = output["in_response_to_tweet_id"].astype("Int64")

grp = output.groupby("conv_id")
assert output["tweet_id"].is_unique
assert (grp["turn_index"].max() == grp.size()).all(), "turn_index not contiguous 1..n"
assert grp["author_id"].apply(lambda s: (s == BRAND).any()).all(), "conversation without brand tweet"
assert output["conv_len"].max() <= MAX_TWEETS

print(f"output {output['conv_id'].nunique():,} conversations | {len(output):,} tweets | "
      f"{len(branch_convs):,} non-linear")
output.head()

output 44,654 conversations | 128,897 tweets | 5,913 non-linear


,conv_id,turn_index,tweet_id,author_id,inbound,created_at,text,in_response_to_tweet_id,response_tweet_id,conv_len,is_linear
0,1,1,699493,286862,True,Sun Apr 02 18:02:25 +0000 2017,@115821 100s of ppl have been scammed from my ...,<NA>,699491,3,True
1,1,2,699491,AmazonHelp,False,Sun Apr 02 18:11:00 +0000 2017,@286862 We're sorry to hear about your account...,699493,699492,3,True
2,1,3,699492,286862,True,Sun Apr 02 18:16:27 +0000 2017,@AmazonHelp That was 3 weeks ago &amp; I recei...,699491,NaN,3,True
3,2,1,381937,206566,True,Wed Sep 27 04:56:45 +0000 2017,"Too many times now, my prime orders have been ...",<NA>,381936,2,True
4,2,2,381936,AmazonHelp,False,Wed Sep 27 05:15:26 +0000 2017,@206566 Apologies for the ordeal. Please let u...,381937,NaN,2,True


In [9]:
output.to_csv(DATA / "amazonhelp_threads.csv", index=False)

acct = pd.DataFrame(steps, columns=["step", "conversations", "tweets"])
acct["conv_%"] = (100 * acct["conversations"] / acct["conversations"].iloc[0]).round(1)
acct["tweet_%"] = (100 * acct["tweets"] / acct["tweets"].iloc[0]).round(1)
print(acct.to_string(index=False))

print(f"\nwrote {len(output):,} rows / {output['conv_id'].nunique():,} conversations "
      "-> amazonhelp_threads.csv")
print("\nturns per conversation:")
print(grp.size().value_counts().sort_index().to_string())

                    step  conversations  tweets  conv_%  tweet_%
AmazonHelp conversations          82556  374042   100.0    100.0
             length <= 5          62962  183744    76.3     49.1
           max gap < 24h          61271  176989    74.2     47.3
         1 customer only          59995  171923    72.7     46.0
              lang == en          44654  128897    54.1     34.5

wrote 128,897 rows / 44,654 conversations -> amazonhelp_threads.csv

turns per conversation:
2    22529
3     8823
4     9140
5     4162


In [10]:
# The reference conversation 2755680 -> 2755678 -> 2755679 has TWO customers: the complainant 771286
# and the third party 771287. Assumption 6 therefore discards it. That makes it a better test than
# before: it checks the grouping and ordering are still right AND that the new filter routed it out of
# the corpus rather than dropping it on the floor.
REF = [2755680, 2755678, 2755679]
ref = third_party[third_party["tweet_id"].isin(REF)].sort_values("ts")

assert len(ref) == 3, f"expected 3 reference tweets in third_party_discard, found {len(ref)}"
assert ref["conv_id"].nunique() == 1, "reference tweets were split across conversations"
assert ref["tweet_id"].tolist() == REF, "reference conversation is in the wrong order"
assert not output["tweet_id"].isin(REF).any(), "reference conversation should not be in the corpus"

print("reference conversation -> third_party_discard.csv, grouped as one, correct order:")
for _, r in ref.iterrows():
    print(f"  {r['author_id']:<11} | {str(r['text'])[:80]}")

# A kept conversation, which by assumption 6 is exactly one customer plus AmazonHelp.
ex = output.loc[output["conv_len"] == 4, "conv_id"].iloc[0]
assert output.loc[output["conv_id"] == ex, "author_id"].nunique() == 2
print(f"\nkept example, conv_id={ex}:")
for _, r in output[output["conv_id"] == ex].iterrows():
    print(f"  turn {r['turn_index']} | {r['author_id']:<11} | {str(r['text'])[:75]}")

reference conversation -> third_party_discard.csv, grouped as one, correct order:
  771286      | Why is the @117634 app for Android incapable of opening to the page on which I l
  AmazonHelp  | @771286 I'm sorry to hear you're having trouble with the Amazon Kindle app. Are 
  771287      | @AmazonHelp @771286 Im having the same problem. I go back and it takes me far ba

kept example, conv_id=11:
  turn 1 | 159148      | @AmazonHelp y does your delivery service bend my book to place in my mailbo
  turn 2 | AmazonHelp  | @159148 Thanks for reaching out to us here! To better assist you, can you c
  turn 3 | 159148      | @AmazonHelp USPS
  turn 4 | AmazonHelp  | @159148 I'm sorry to hear that, contact us here:  https://t.co/hApLpMlfHN w


## `Data/amazonhelp_threads.csv`

One row per tweet, grouped by `conv_id`, ordered by `turn_index`.

| column | meaning |
|---|---|
| `conv_id` | conversation id, numbered from 1 by the conversation's first tweet (reconstructed, not in raw data) |
| `turn_index` | position in the conversation, 1..`conv_len` |
| `tweet_id`, `author_id`, `inbound`, `created_at`, `text`, `response_tweet_id` | carried over unchanged |
| `in_response_to_tweet_id` | parent tweet, null at turn 1. Recovers the true tree for branching conversations |
| `conv_len` | turns in this conversation (2–5) |
| `is_linear` | False when some tweet in the conversation received more than one reply |

Every kept conversation has 2–5 turns, completes within 24 hours, and has exactly two authors — one customer and AmazonHelp. There is no `n_authors` column because after assumption 6 it is always 2.

## Discarded conversations

The four files are disjoint and together account for everything dropped after grouping:

| file | rule | extra column |
|---|---|---|
| `length_discard.csv` | more than 5 tweets (assumption 4) | — |
| `gap_discard.csv` | a gap of 24 hours or more between turns (assumption 5) | `max_gap_hours` |
| `third_party_discard.csv` | more than one customer, or none (assumption 6) | `n_customers` |
| `lang_discard.csv` | customer text not detected as English (assumption 7) | `lang`, `lang_conf` |

All four carry the raw columns plus `conv_len`, so a discarded conversation can be read and judged directly rather than taken on trust. The accounting table should reconcile: kept + the four discards = the AmazonHelp conversation count from the grouping step.

## Limitations

- **Not a representative sample, and the filters compound.** ≤5 turns removes long threads, <24 hours removes slow ones, single-customer removes popular ones, English-only removes whole markets. Between them that is much of what makes real support hard. The accounting table gives the cost of each.
- **The single-customer rule removes attention, not just ambiguity.** The ~2% dropped are the threads strangers joined, which correlates with problems many people had. A widely-shared bug is exactly the kind of issue a support system should handle, not the kind it should be evaluated without.
- **The 24-hour gap cap is the strictest option considered** — roughly 3% of conversations versus ~0.6% for a 7-day cap. Most of that difference is genuine slow support. Sort `gap_discard.csv` by `max_gap_hours` ascending to see what is lost at the margin.
- `langdetect` is weak on short text. Tweets are short, emoji-heavy and code-switched. Spot-check `lang_discard.csv` — especially `lang_conf` on one-line tweets — before quoting the English count.
- **Branching conversations read as interleaved.** `turn_index` is chronological, so where `is_linear == False`, consecutive turns are not necessarily replies to one another. Use `in_response_to_tweet_id` when the real reply structure matters.
- **Multi-part replies stay separate rows.** AmazonHelp splits answers to fit the character limit (`(1/2)`, `(2/2)`), so one row is not always one complete reply.
- **Thread membership does not imply a support request.** Community-manager banter survives every filter.
- **Nothing here observes resolution** — the dataset has no ticket status or outcome.